In [3]:
import pandas as pd
import numpy as np
from pathlib import Path

In [4]:
STOCK_DATA_PATH  = Path('data/stock_data_full.csv')
SENTIMENT_PATH   = Path('../sentiment/data/all_labeled_finnishbert_new.parquet')
FORUM_POSTS_PATH = Path('../sentiment/data/cleaned_forum_posts.parquet')
OUTPUT_PATH      = Path('data/panel_data.parquet')

stock_data   = pd.read_csv(STOCK_DATA_PATH, parse_dates=['Date'])
sentiment    = pd.read_parquet(SENTIMENT_PATH, columns=['id', 'sentiment'])
forum_posts  = pd.read_parquet(FORUM_POSTS_PATH, columns=['id', 'date_time', 'ticker'])

print(f'Stock data   : {len(stock_data):,} rows')
print(f'Sentiment    : {len(sentiment):,} rows')
print(f'Forum posts  : {len(forum_posts):,} rows')

Stock data   : 390,127 rows
Sentiment    : 532,424 rows
Forum posts  : 532,424 rows


## Build prev_day_sentiment

In [5]:
# Join sentiment labels onto forum posts to get date + ticker per post
sent_dated = sentiment.merge(forum_posts, on='id', how='inner')
sent_dated['date'] = sent_dated['date_time'].dt.normalize()  # strip time, keep date

# Aggregate to mean sentiment per ticker per calendar day
daily_sent = (
    sent_dated
    .groupby(['ticker', 'date'], sort=False)['sentiment']
    .mean()
    .reset_index()
    .rename(columns={'date': 'Date', 'sentiment': 'daily_sentiment'})
)

print(f'Daily sentiment rows : {len(daily_sent):,}')
print(f'Tickers with sentiment: {daily_sent["ticker"].nunique()}')
print(daily_sent.head())

Daily sentiment rows : 88,986
Tickers with sentiment: 162
  ticker       Date  daily_sentiment
0  ADMCM 2018-02-02         1.166667
1  ADMCM 2018-02-08         1.500000
2  ADMCM 2018-02-09         1.285714
3  ADMCM 2018-02-12         1.142857
4  ADMCM 2018-05-09         0.000000


In [6]:
# Shift sentiment forward 1 calendar day so that posts on day D
# appear as prev_day_sentiment on day D+1 in the stock data.
daily_sent['lookup_date'] = daily_sent['Date'] + pd.Timedelta(days=1)

stock_data = stock_data.merge(
    daily_sent[['ticker', 'lookup_date', 'daily_sentiment']].rename(
        columns={'lookup_date': 'Date', 'daily_sentiment': 'prev_day_sentiment'}
    ),
    on=['ticker', 'Date'],
    how='left',
)

stock_data['prev_day_sentiment'] = stock_data['prev_day_sentiment'].fillna(1)

print(stock_data[['ticker', 'Date', 'prev_day_sentiment']].head(10))

   ticker       Date  prev_day_sentiment
0  AALLON 2019-04-08            1.000000
1  AALLON 2019-04-09            0.944444
2  AALLON 2019-04-10            0.750000
3  AALLON 2019-04-11            1.000000
4  AALLON 2019-04-12            1.000000
5  AALLON 2019-04-15            1.000000
6  AALLON 2019-04-16            1.000000
7  AALLON 2019-04-17            2.000000
8  AALLON 2019-04-18            0.500000
9  AALLON 2019-04-23            1.000000


## Save

In [7]:
stock_data.to_parquet(OUTPUT_PATH, index=False, compression='brotli')
print(f'Saved {len(stock_data):,} rows to {OUTPUT_PATH}')
print(f'Columns: {stock_data.columns.tolist()}')

Saved 390,127 rows to data/panel_data.parquet
Columns: ['ticker', 'Date', 'return', 'return_lag1', 'return_lag2', 'return_lag5', 'log_volume', 'gk_vol', 'momentum_12_1', 'amihud', 'open_to_close_return', 'close_to_open_return', 'EURUSD_return', 'EURSEK_return', 'EURCNY_return', 'GC_return', 'CL_return', 'VIX_return', 'GSPC_return', 'TNX_return', 'euribor_3m_diff', 'OMXHPI_return', 'OMXN40_return', 'STOXX50E_return', 'finland_10y_diff', 'unemp_rate_change_lagged', 'cpi_rate_change_lagged', 'consumer_conf_change_lagged', 'prev_day_sentiment']


In [8]:
stock_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 390127 entries, 0 to 390126
Data columns (total 29 columns):
 #   Column                       Non-Null Count   Dtype         
---  ------                       --------------   -----         
 0   ticker                       390127 non-null  object        
 1   Date                         390127 non-null  datetime64[ns]
 2   return                       389965 non-null  float64       
 3   return_lag1                  389803 non-null  float64       
 4   return_lag2                  389641 non-null  float64       
 5   return_lag5                  389155 non-null  float64       
 6   log_volume                   390127 non-null  float64       
 7   gk_vol                       390127 non-null  float64       
 8   momentum_12_1                350240 non-null  float64       
 9   amihud                       389803 non-null  float64       
 10  open_to_close_return         390127 non-null  float64       
 11  close_to_open_return      